<img src="../../../Individual-Contest/Antique/figs/IOAI-Logo.png" alt="Logo IOAI" width="200" height="auto">

[IOAI 2025 (Pekin, Chine), concours individuel](https://ioai-official.org/china-2025)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SKonteye/IOAI-2025/blob/main/fr/Individual-Contest/Antique/Antique.ipynb)


# Authentification de peintures anciennes

## 1. Description du probleme

Vous etudiez l'intelligence artificielle depuis un certain temps. Un vieil ami de votre pere, archeologue et critique d'art renomme, a entendu parler de vous et vous demande de l'aide. Vous devez concevoir un algorithme capable de classifier des peintures anciennes en tant qu'authentiques ou copies.

L'authentification professionnelle etant couteuse, l'equipe de recherche n'a obtenu des etiquettes d'authenticite que pour une faible partie des peintures. Pour la majorite des echantillons, l'authenticite reste inconnue. On sait que les caracteristiques numeriques des peintures presentent de fortes structures. Vous devez exploiter l'ensemble des echantillons disponibles, y compris ceux dont l'etiquette est inconnue, pour entrainer un modele permettant de classifier l'authenticite des peintures anciennes.

## 2. Jeu de donnees

Le jeu de donnees comprend un ensemble d'entrainement, un ensemble de validation et un ensemble de test, chacun comportant 500 echantillons independants.

1. **Ensemble d'entrainement (`training_set.csv`)** :

   - Les cinq premieres colonnes representent les caracteristiques numeriques de chaque peinture ancienne.
   - La sixieme colonne contient le label : 1 pour authentique, -1 pour copie, et 0 pour inconnu.

   L'ensemble d'entrainement sert a entrainer vos modeles et peut etre consulte et telecharge directement pendant le concours.

2. **Ensemble de validation (`validation_set.csv`)** :
   - Format similaire a l'ensemble d'entrainement, mais sans la colonne de label.

   L'ensemble de validation est utilise pour calculer le score du classement A et n'est pas directement accessible pendant le concours.

3. **Ensemble de test (`test_set.csv`)** :
   - Format similaire a l'ensemble d'entrainement, mais sans la colonne de label.

   L'ensemble de test sert a calculer le score du classement B et n'est pas directement accessible pendant le concours.

## 3. Tache

Votre tache est d'entrainer un modele approprie capable de predire l'authenticite des peintures dans les ensembles de test, malgre le grand nombre d'echantillons non etiquetes.

## 4. Soumission

Les concurrents doivent soumettre un fichier notebook nomme `submission.ipynb`. Ce fichier doit produire une archive zip nommee `submission.zip` contenant les deux fichiers suivants :

1. `submissionA.csv` : contient les predictions du modele sur l'ensemble de validation, une valeur -1 ou 1 par ligne, sans entete.
2. `submissionB.csv` : contient les predictions du modele sur l'ensemble de test, une valeur -1 ou 1 par ligne, sans entete.

La machine de test lit `submission.zip` et calcule les scores. Les fichiers de soumission doivent respecter strictement le format et le nommage ci-dessus, faute de quoi le systeme ne pourra pas les lire correctement.

Les details de la procedure de soumission figurent dans le notebook baseline. Les concurrents sont invites a s'y referer.

## 5. Score

La metrique d'evaluation est la **precision de classification**, definie comme la proportion d'echantillons correctement predits parmi le total des echantillons evalues.

## 6. Baseline et ensemble d'entrainement

- Vous trouverez ci-dessous la solution baseline.
- Le jeu de donnees se trouve dans le dossier `training_set`.
- Le meilleur score du comite scientifique pour cette tache est 0.98 sur le classement B ; ce score est utilise pour l'unification des scores.
- Le score baseline du comite scientifique pour cette tache est 0.46 sur le classement B ; ce score est utilise pour l'unification des scores.


### Entrainez votre modele


In [ ]:
import os
import sys

# 1. Recupere le repertoire de travail courant
current_dir = os.getcwd()

# 2. Verifie que le chemin contient "Individual-Contest/Antique" et le tronque a cet endroit
if "Individual-Contest/Antique" in current_dir:
    root_index = current_dir.index("Individual-Contest/Antique") + len("Individual-Contest/Antique")
    project_root = current_dir[:root_index]
else:
    raise Exception("Repertoire racine du projet introuvable. Verifiez la structure des dossiers.")

# 3. Change le repertoire de travail vers la racine du projet
os.chdir(project_root)
print("Repertoire de travail defini sur :", os.getcwd())

# 4. Ajoute le chemin de recherche des modules (par exemple, l'emplacement de metrics.py)
sys.path.append(os.path.join(project_root, "Scoring"))

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.svm import SVC

TRAIN_PATH = "./training_set/"
train = pd.read_csv(TRAIN_PATH + "training_set.csv")

X = np.array(train.iloc[:,:5])
y = np.array(train.iloc[:,5])

np.random.seed(42)
y[y == 0] = np.random.choice([-1, 1], size=(y == 0).sum())

svm_binary_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_binary_model.fit(X, y)

### Faites les predictions sur les ensembles de validation et de test


In [ ]:
VAL_DATA_PATH = "./Solution/validation_set/"
TEST_DATA_PATH = "./Solution/test_set/"

testA = np.array(pd.read_csv(VAL_DATA_PATH + "validation_set.csv"))
testB = np.array(pd.read_csv(TEST_DATA_PATH + "test_set.csv"))

predA = svm_binary_model.predict(testA)
predB = svm_binary_model.predict(testB)

### Generez `submission.zip` pour la soumission


In [ ]:
import zipfile
import os

submissionA = pd.DataFrame(predA)
submissionA.to_csv("./Scoring/submissionA.csv", index=False, header=False)

submissionB = pd.DataFrame(predB)
submissionB.to_csv("./Scoring/submissionB.csv", index=False, header=False)

files_to_zip = ['./Scoring/submissionA.csv', './Scoring/submissionB.csv']
zip_filename = './Scoring/submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} a ete cree avec succes !')

### Evaluez les performances du modele


In [ ]:
%run Scoring/metrics.py